# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = 'https://raw.githubusercontent.com/sidluthra1/DS2002FA26/main/notebooks/06-midterm-walmart/data/product_catalog.csv'  # Colab: read the real midterm data from the repo
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
pop = cat[cat['product_name'].str.lower().str.contains('pop')]
print(len(pop))
pop

4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


I lowercased the names before searching for 'pop' so spellings like "Pop-Tarts" and "pop-tarts" are both counted.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
straw = pop[pop['product_name'].str.lower().str.contains('strawberry')]
print(straw[['sku', 'product_name']])
print(pop[~pop.index.isin(straw.index)])  # Pop-Tart rows that are NOT strawberry

             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  pop-tarts strawberry
Empty DataFrame
Columns: [sku, product_name, category, subcategory, unit_cost, supplier, package_size]
Index: []


I split the Pop-Tart rows on 'strawberry' so the strawberry variants are listed and any non-strawberry Pop-Tart rows show up separately instead of being merged.

**Pop-Tart rows that are a different product:** None — all 4 Pop-Tart rows (PT-12, SKU#459812, POPTART-STRAW, PT_STRAWBERRY) are strawberry, so the non-strawberry filter returns an empty frame and there is nothing that must stay separate.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
sku_map = {'SKU#459812': 'PT-12', 'POPTART-STRAW': 'PT-12', 'PT_STRAWBERRY': 'PT-12'}
cat_mapped = cat.assign(canonical_sku=cat['sku'].map(sku_map).fillna(cat['sku']))
cat_mapped[['sku', 'canonical_sku']].head(6)

,sku,canonical_sku
0,PT-12,PT-12
1,SKU#459812,PT-12
2,POPTART-STRAW,PT-12
3,PT_STRAWBERRY,PT-12
4,WTR-24,WTR-24
5,WATER24PK,WATER24PK


I mapped every Pop-Tart variant to PT-12 and used .map(...).fillna(cat['sku']) so SKUs not in the dict keep their own value instead of becoming NaN.

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print(cat['category'].value_counts(), '\n')
cat_mapped['category_clean'] = cat_mapped['category'].str.strip().str.title()
print(cat_mapped['category_clean'].value_counts())

category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64 

category_clean
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast Foods        

I stripped and title-cased to collapse case/whitespace duplicates (e.g., EMERGENCY/emergency/Emergency → Emergency); still two labels for one thing: Breakfast vs Breakfast Foods, Emergency vs Emergency Supplies, Grocery vs Groceries, Alcohol vs Alcoholic Beverages, and Baby vs Baby Products.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
import re
def name_key(s):
    return ' '.join(sorted(re.sub(r'[^\w\s]', ' ', s.lower()).split()))

groups = cat.assign(name_key=cat['product_name'].apply(name_key)).groupby('name_key')['sku'].agg(['nunique', list])
groups[groups['nunique'] > 1]

,nunique,list
name_key,,
24 beer light pack,2,"[BEER-24, LTBEER24]"
4 aa batteries pack,2,"[BAT-AA4, BATT_AA_4PK]"
aid first kit,2,"[FAID-01, FIRSTAID-KIT]"
board game monopoly,3,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,2,"[BREAD-WH, WHITEBREAD]"
butter peanut,2,"[PB-16, PNUTBTR-16]"
candles emergency,2,"[CNDL-EM, CANDLE-EMRG]"
canned soup,2,"[SOUP-CAN, CNSOUP-01]"
cards playing,2,"[CARDS-01, PLAYCARD-STD]"


I keyed each name on its lowercased, punctuation-free, sorted words so variants like "Pop-Tarts Strawberry" and "pop-tarts strawberry" collide, then kept keys with more than one SKU as fragmentation candidates.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
assert cat_mapped['canonical_sku'].notna().all(), 'mapping produced nulls'
assert len(cat_mapped) == len(cat), 'mapping changed the row count'
print('Mapping is safe:', len(cat_mapped), 'rows, 0 null canonical SKUs')

Mapping is safe: 45 rows, 0 null canonical SKUs


I asserted no nulls and an unchanged row count because a correct consolidation should only rename SKUs, never add or drop rows.

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**a)** Take AA batteries and 24-pack bottled water, for example. In order to clean rgem up a bit, would utilize a name key, like in Q5, I would also do a small synonym step, like changing "pk to pack" and "24pk" to "24 pack", and getting rid of unnecessary words like "bottled". This way, all the different versions can be put into one category. After a review, each group can be matched to a single, standard product code.

**b)** When the copies are different in certain aspects that matter to the business, it is considered a duplicate. If you merge them, what is ordered and reported change. An example i'm not entirely sure about I'm unsure about is STREAM-01 "Roku Streaming Stick" vs. ROKU-STD "Streaming Device", both are in the same subcategory and similar in cost, but one's a named brand and the other's generic, so they might actually be different products. It's kind of like the Pop-Tart rows, where we've got 4-packs and 6-packs from different suppliers all mixed together. In this case, I'd check with the person in charge of electronics or the product data owner to see if they think they're the same thing before consolidating them.